# Módulo 09 · Aula 02 — Deploy Prático

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O deploy é assim: eu entro no servidor por SSH com a senha de root, faço `git pull`, mato o processo com `kill`, e subo de novo com `nohup`. Se der errado, eu... não sei. Da última vez a gente restaurou de um backup de três dias antes."*

Cada frase dessa tem um problema. Vamos resolver todos.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | SSH com chave | 🔴 Senha de root não |
| 2 | Usuário de deploy | Menor privilégio |
| 3 | **`releases/` + symlink** | 🎯 Troca atômica |
| 4 | `rsync` | Enviar só o necessário |
| 5 | `systemd` | Quem reinicia quando cai |
| 6 | 🔴 **Migrações no deploy** | A ordem que evita queda |
| 7 | **Rollback** | O que você vai querer às 2h |
| 8 | Deploy sem queda | E os limites disso |
| 9 | PaaS | Quando não vale ter servidor |

> 🎯 **O padrão de deploy roda ao vivo aqui:** um "servidor" local com `releases/`, symlink atômico, `rsync` de verdade e rollback em um comando. O que muda num servidor real é o endereço.

## ⚙️ Preparação

| | O que é | Como aparece |
|---|---------|--------------|
| ✅ **Executado** | `ssh-keygen`, `rsync`, symlinks atômicos, validação de unit `systemd` | Saída normal |
| 📖 **Referência** | Comandos contra um servidor remoto | `[referência]` |

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 09
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import re
import shutil
import socket
import subprocess
import sys
import textwrap
import threading
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn"), ("pyyaml", "yaml"),
               ("gunicorn", "gunicorn")]:
    _garantir(_p, _m)

import httpx
import uvicorn
import yaml
from fastapi import FastAPI

TEM_GUNICORN = _garantir("gunicorn", "gunicorn")
E_LINUX = platform.system() == "Linux"

print(f"sistema   : {platform.system()}")
print(f"gunicorn  : {'✅ disponível' if TEM_GUNICORN else '⚠️ indisponível (só Unix)'}")


# ═══════════════════════════════════════════════════════════════
#  Servidores de verdade
# ═══════════════════════════════════════════════════════════════

def porta_livre() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    """Sobe uma app ASGI num uvicorn de verdade, em segundo plano."""

    def __init__(self, app, nome: str = "servico", **config):
        self.app, self.nome = app, nome
        self.porta = porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self.config = config
        self._servidor = self._thread = None

    def iniciar(self, timeout: float = 20.0) -> "Servico":
        cfg = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                             log_level="critical", access_log=False, **self.config)
        self._servidor = uvicorn.Server(cfg)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()
        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)

    def __enter__(self):
        return self.iniciar()

    def __exit__(self, *_):
        self.parar()


# ═══════════════════════════════════════════════════════════════
#  Shell e exibição
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120,
       env: dict | None = None) -> subprocess.CompletedProcess:
    p = subprocess.run(comando, shell=True, capture_output=True, text=True,
                       cwd=cwd, timeout=timeout,
                       env=dict(os.environ, **(env or {})))
    if mostrar:
        saida = (p.stdout + p.stderr).rstrip()
        if saida:
            print(saida)
    return p


def referencia(comando: str, esperado: str = "") -> None:
    """Mostra um comando que NÃO roda aqui, com a saída típica.

    🔴 Saída marcada `[referência]` não foi executada. Rode você mesmo —
       é assim que se aprende deploy.
    """
    print(f"$ {comando}")
    if esperado:
        for linha in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {linha}")
    print("  ── [referência] não executado neste ambiente ──")


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]) -> None:
    """Tabela ASCII alinhada (marcadores ASCII, não emoji — M03)."""
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = "") -> None:
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        print(f"{prefixo}{'└── ' if ultimo else '├── '}{item.name}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


print("✅ `Servico`, `sh()`, `referencia()`, `tabela()`, `preparar()` prontos")

## 1. O que está errado no deploy da Aurora

In [ ]:
BASE = preparar("aula_09_02")

problemas = [
    ["ssh root@servidor",   "🔴 login como root com senha",
     "chave + usuário sem privilégio"],
    ["git pull",            "🔴 build acontece em produção",
     "envie o ARTEFATO pronto"],
    ["kill <pid>",          "🔴 corta requisições em andamento",
     "SIGTERM e carência (09_01)"],
    ["nohup ... &",         "🔴 ninguém reinicia se cair",
     "systemd ou Docker"],
    ["(sem rollback)",      "🔴 restaurar backup de 3 dias",
     "release anterior a um comando"],
    ["(sem migração)",      "🔴 esquecer quebra o site",
     "passo explícito e ordenado"],
]
tabela(["O QUE FAZEM", "PROBLEMA", "O QUE FAZER"], problemas, [20, 34, 32])

print("""
💭 O `git pull` merece um parágrafo.

   Ele parece inofensivo e é o mais perigoso da lista. Fazer build em
   produção significa que o que roda lá nunca foi testado: uma
   dependência que resolveu diferente, um arquivo não commitado, um
   `.pyc` velho.

   🎯 A regra é a mesma do M08: CONSTRUA UMA VEZ, PROMOVA O ARTEFATO.
      O que passou na homologação é bit a bit o que sobe.
""")

## 2. SSH com chave

In [ ]:
CHAVES = BASE / "chaves"
CHAVES.mkdir()

# Geramos um par de verdade
p = sh(f'ssh-keygen -t ed25519 -f "{CHAVES}/atlas_deploy" -N "" '
       f'-C "deploy@atlas" -q', mostrar=False)

publica = (CHAVES / "atlas_deploy.pub").read_text(encoding="utf-8").strip()
print("✅ par de chaves gerado\n")
print(f"pública  ({len(publica)} chars):")
print(f"   {publica}\n")

privada = (CHAVES / "atlas_deploy").read_text(encoding="utf-8")
print(f"privada  ({len(privada)} chars): 🔴 NUNCA sai da sua máquina")
print(f"   {privada.splitlines()[0]}")
print("   ... (conteúdo omitido de propósito)")
print(f"   {privada.splitlines()[-1]}\n")

sh(f'ssh-keygen -lf "{CHAVES}/atlas_deploy.pub"')

> 🔑 **Ed25519, não RSA.** Chaves menores, mais rápidas, e sem a armadilha do tamanho — uma RSA de 1024 bits ainda funciona e não deveria.
>
> 🔴 **A privada nunca sai da sua máquina.** Nem por e-mail, nem no Slack, nem no repositório. O que você copia para o servidor é a **pública** — e ela é pública mesmo, dá para publicar no GitHub.
>
> 💡 **Use senha na chave** (`-N` vazio acima só para o exemplo rodar). Com `ssh-agent`, você digita uma vez por sessão.

In [ ]:
# ~/.ssh/config — o arquivo que economiza mais digitação no ano
CONFIG_SSH = BASE / "config_ssh_exemplo"
CONFIG_SSH.write_text("""# ~/.ssh/config

Host atlas-prod
    HostName 203.0.113.10
    User deploy
    IdentityFile ~/.ssh/atlas_deploy
    IdentitiesOnly yes
    Port 22

Host atlas-homolog
    HostName 203.0.113.11
    User deploy
    IdentityFile ~/.ssh/atlas_deploy
    IdentitiesOnly yes

# 💡 `IdentitiesOnly yes` evita que o ssh ofereça TODAS as suas chaves.
#    Sem isso, com muitas chaves você leva "Too many authentication
#    failures" antes de chegar na certa.
""", encoding="utf-8")

print(CONFIG_SSH.read_text(encoding="utf-8"))
print("Agora basta:  ssh atlas-prod")

In [ ]:
referencia("ssh-copy-id -i ~/.ssh/atlas_deploy.pub deploy@203.0.113.10", """
Number of key(s) added: 1
Now try logging into the machine, with:   "ssh 'deploy@203.0.113.10'"
""")

print()
print("🔴 E DEPOIS, ENDUREÇA O SERVIDOR — /etc/ssh/sshd_config:\n")
ENDURECER = """
PermitRootLogin no              # 🔴 ninguém entra como root
PasswordAuthentication no       # 🔴 só chave; mata ataque de dicionário
PubkeyAuthentication yes
AllowUsers deploy               # só este usuário
MaxAuthTries 3
"""
for linha in ENDURECER.strip().splitlines():
    print(f"   {linha}")

print("\n⚠️ ANTES de `systemctl restart sshd`, abra uma SEGUNDA sessão SSH")
print("   e deixe-a aberta. Se você errar a configuração, ela é a sua")
print("   única forma de voltar atrás — a sessão já aberta sobrevive ao")
print("   restart, uma nova não conseguiria entrar.")
print("\n   💭 Todo mundo aprende isso da forma difícil, uma vez.")

## 3. Usuário de deploy

In [ ]:
referencia("""sudo adduser --disabled-password --gecos "" deploy
sudo mkdir -p /opt/atlas /etc/atlas
sudo chown -R deploy:deploy /opt/atlas
sudo chmod 750 /etc/atlas""", """
# 🔑 `--disabled-password`: este usuário NÃO tem senha.
#    Só entra por chave — não há senha para adivinhar.
""")

print()
print("🔒 E o mínimo de sudo, só para o que o deploy precisa:\n")
print("   # /etc/sudoers.d/deploy")
print("   deploy ALL=(root) NOPASSWD: /bin/systemctl restart atlas")
print("   deploy ALL=(root) NOPASSWD: /bin/systemctl status atlas")
print("""
🎯 Repare no que ISSO IMPEDE: se a chave de deploy vazar, o atacante
   consegue reiniciar o Atlas — e mais nada. Não instala pacote, não
   lê /etc/shadow, não vira root.

   💭 É o mesmo princípio dos papéis do M06: dê o mínimo necessário,
      e o estrago de um comprometimento fica contido.
""")

## 4. 🎯 O padrão `releases/` + symlink

In [ ]:
print("""
   /opt/atlas/
   ├── releases/
   │   ├── 20260810-a1b2c3/      versão antiga
   │   ├── 20260812-d4e5f6/      versão anterior   ← rollback vai para cá
   │   └── 20260813-9f8e7d/      versão nova
   ├── compartilhado/
   │   ├── .env                  🔑 config fora do release
   │   └── uploads/              🔑 dados fora do release
   └── atual -> releases/20260813-9f8e7d      ← 🎯 UM SYMLINK

   O deploy é: descompacta em releases/, e TROCA O SYMLINK.
   O rollback é: TROCA O SYMLINK DE VOLTA.
""")

# Montamos um "servidor" de verdade, aqui mesmo
SERVIDOR = BASE / "servidor"
(SERVIDOR / "releases").mkdir(parents=True)
(SERVIDOR / "compartilhado").mkdir()
(SERVIDOR / "compartilhado" / ".env").write_text(
    "ATLAS_AMBIENTE=producao\nATLAS_SECRET_KEY=chave-real-do-servidor\n",
    encoding="utf-8")

print("🟢 servidor de mentira criado em", SERVIDOR.name)

In [ ]:
import os
from datetime import datetime, timedelta, timezone


def publicar(versao: str, conteudo: str) -> Path:
    """Cria um release e aponta o symlink para ele — ATOMICAMENTE."""
    destino = SERVIDOR / "releases" / versao
    destino.mkdir(parents=True, exist_ok=True)
    (destino / "app.py").write_text(conteudo, encoding="utf-8")

    # 🔑 O .env NÃO vive no release — ele é um link para o compartilhado.
    #    Assim a configuração sobrevive a todo deploy e a todo rollback.
    ligacao = destino / ".env"
    if not ligacao.exists():
        os.symlink("../../compartilhado/.env", ligacao)

    # 🎯 A TROCA ATÔMICA
    #
    #    `ln -sfn` NÃO é atômico: ele remove e recria, e existe uma
    #    janela de microssegundos em que `atual` não aponta para nada.
    #    Sob carga, alguém cai nessa janela.
    #
    #    O jeito certo: criar um link temporário e RENOMEAR por cima.
    #    O `rename` é atômico no POSIX — ou é o antigo, ou é o novo,
    #    nunca um estado intermediário.
    temporario = SERVIDOR / "atual.novo"
    if temporario.exists() or temporario.is_symlink():
        temporario.unlink()
    os.symlink(f"releases/{versao}", temporario)
    os.replace(temporario, SERVIDOR / "atual")
    return destino


def situacao():
    atual = os.readlink(SERVIDOR / "atual")
    app = (SERVIDOR / "atual" / "app.py").read_text(encoding="utf-8").strip()
    env = (SERVIDOR / "atual" / ".env").read_text(encoding="utf-8").splitlines()[0]
    print(f"   atual  → {atual}")
    print(f"   app.py : {app}")
    print(f"   .env   : {env}  (vem do compartilhado)")


agora = datetime.now(timezone.utc)
v1 = (agora - timedelta(days=3)).strftime("%Y%m%d") + "-a1b2c3"
v2 = (agora - timedelta(days=1)).strftime("%Y%m%d") + "-d4e5f6"

publicar(v1, "VERSAO = 'antiga'")
print("Depois do primeiro deploy:")
situacao()

publicar(v2, "VERSAO = 'anterior'")
print("\nDepois do segundo deploy:")
situacao()

> 🎯 **`os.replace()` sobre o symlink é o coração do deploy sem queda.**
>
> A troca é atômica: qualquer processo que abrir `atual/` naquele instante pega ou a versão antiga inteira, ou a nova inteira. Nunca um pedaço de cada.
>
> ⚠️ **`ln -sfn` não dá essa garantia** — ele apaga e recria. Sob carga, alguém vai cair na janela de microssegundos em que o link não existe. É o tipo de bug que acontece uma vez a cada dez mil requisições e ninguém consegue reproduzir.
>
> 🔑 **E repare no `.env`:** ele mora em `compartilhado/` e cada release o referencia. Sem isso, todo deploy precisaria recriar a configuração — e um rollback voltaria para uma configuração antiga junto com o código.

## 5. `rsync` — enviando só o necessário

In [ ]:
# O que sai da sua máquina
LOCAL = BASE / "local"
(LOCAL / "src" / "atlas").mkdir(parents=True)
(LOCAL / "src" / "atlas" / "api.py").write_text("app = 'nova'\n", encoding="utf-8")
(LOCAL / "src" / "atlas" / "__pycache__").mkdir()
(LOCAL / "src" / "atlas" / "__pycache__" / "api.cpython.pyc").write_bytes(b"\x00" * 500)
(LOCAL / ".venv").mkdir()
(LOCAL / ".venv" / "grande.bin").write_bytes(b"\x00" * 200_000)
(LOCAL / ".git").mkdir()
(LOCAL / ".git" / "objects.bin").write_bytes(b"\x00" * 100_000)
(LOCAL / ".env").write_text("ATLAS_SECRET_KEY=chave-de-desenvolvimento\n",
                            encoding="utf-8")
(LOCAL / "pyproject.toml").write_text("[project]\nname='atlas'\n", encoding="utf-8")
(LOCAL / "tests").mkdir()
(LOCAL / "tests" / "test_api.py").write_text("def test_x(): pass\n", encoding="utf-8")

EXCLUSOES = BASE / "deploy_excluir.txt"
EXCLUSOES.write_text("""# 🔴 A MESMA lista do .dockerignore, e pelo mesmo motivo.
.env
.venv/
.git/
__pycache__/
*.pyc
tests/
notebooks/
saida/
""", encoding="utf-8")

antes = sum(f.stat().st_size for f in LOCAL.rglob("*") if f.is_file())

v3 = agora.strftime("%Y%m%d") + "-9f8e7d"
DESTINO = SERVIDOR / "releases" / v3
DESTINO.mkdir(parents=True)

r = sh(f'rsync -a --delete --exclude-from="{EXCLUSOES}" '
       f'"{LOCAL}/" "{DESTINO}/"', mostrar=False)

depois = sum(f.stat().st_size for f in DESTINO.rglob("*") if f.is_file())

def tamanho(n: int) -> str:
    return f"{n:,} B" if n < 10_000 else f"{n / 1024:,.1f} KB"


print(f"origem  : {tamanho(antes):>12}")
print(f"enviado : {tamanho(depois):>12}")
print(f"economia: {(1 - depois / antes) * 100:>11.1f}%\n")

print("O que chegou no servidor:")
for f in sorted(DESTINO.rglob("*")):
    if f.is_file():
        print(f"   {str(f.relative_to(DESTINO)):<28} {tamanho(f.stat().st_size):>10}")

print("\nO que ficou para trás:")
for f in sorted(LOCAL.rglob("*")):
    if f.is_file() and not (DESTINO / f.relative_to(LOCAL)).exists():
        print(f"   {str(f.relative_to(LOCAL)):<28} {tamanho(f.stat().st_size):>10}")

print("\n🔴 Repare no que NÃO chegou: `.env` com a chave de")
print("   desenvolvimento. Ele existe no servidor — mas em")
print("   `compartilhado/`, com a chave DE PRODUÇÃO.")

In [ ]:
print("""
💡 AS OPÇÕES DO RSYNC QUE IMPORTAM

   -a              arquivo: preserva permissão, data, links
   --delete        🔴 apaga no destino o que não existe na origem
   --exclude-from  a lista de exclusões
   -z              comprime na transferência (bom em rede lenta)
   --dry-run       🔑 mostra o que FARIA, sem fazer
   --stats         resumo do que foi transferido

🔴 `--delete` é poderoso e perigoso. SEMPRE rode com `--dry-run`
   primeiro na primeira vez que apontar para um destino novo. Um
   caminho errado com `--delete` apaga o destino inteiro.

💡 E o rsync só transfere o que MUDOU, bloco a bloco. Num deploy
   típico, isso são alguns KB mesmo com um projeto de 50 MB.
""")

referencia("rsync -az --delete --exclude-from=deploy_excluir.txt "
           "./ deploy@atlas-prod:/opt/atlas/releases/20260813-9f8e7d/", """
sending incremental file list
src/atlas/api.py
sent 4.212 bytes  received 95 bytes  8.614,00 bytes/sec
total size is 182.334  speedup is 42,34
""")

## 6. `systemd` — quem cuida do processo

In [ ]:
UNIT = BASE / "atlas.service"
UNIT.write_text("""[Unit]
Description=Atlas API — Aurora Comércio
Documentation=https://github.com/aurora/atlas
After=network-online.target postgresql.service
Wants=network-online.target

[Service]
Type=exec
User=deploy
Group=deploy

WorkingDirectory=/opt/atlas/atual
EnvironmentFile=/etc/atlas/atlas.env

ExecStart=/opt/atlas/atual/.venv/bin/gunicorn \\
    atlas.api.aplicacao:criar_app \\
    -k uvicorn.workers.UvicornWorker \\
    -w 4 -b 127.0.0.1:8000 \\
    --graceful-timeout 30 \\
    --forwarded-allow-ips 127.0.0.1 \\
    --access-logfile - --error-logfile -

# 🔑 Recarrega os workers SEM derrubar o serviço
ExecReload=/bin/kill -s HUP $MAINPID

# ── Reinício automático ──
Restart=always
RestartSec=5
# 🔴 Sem isto, um erro na partida vira laço infinito de reinício que
#    consome CPU e enche o log. Com isto, o systemd desiste e marca
#    o serviço como `failed` — onde o seu monitoramento o encontra.
StartLimitIntervalSec=300
StartLimitBurst=5

# ── Encerramento (09_01) ──
KillSignal=SIGTERM
TimeoutStopSec=45
# 🔑 Maior que o --graceful-timeout do gunicorn, senão o systemd mata
#    antes de o gunicorn terminar de encerrar com educação.

# ── Endurecimento ──
NoNewPrivileges=true
PrivateTmp=true
ProtectSystem=strict
ProtectHome=true
ReadWritePaths=/opt/atlas/compartilhado
# 🔒 ProtectSystem=strict deixa TODO o sistema de arquivos só-leitura,
#    exceto o que você liberar. Se a aplicação for comprometida, ela
#    não escreve em /etc nem em /usr.

[Install]
WantedBy=multi-user.target
""", encoding="utf-8")

print(f"✅ atlas.service ({len(UNIT.read_text(encoding='utf-8').splitlines())} linhas)\n")

# Validação DE VERDADE
if shutil.which("systemd-analyze"):
    print("── systemd-analyze verify (execução real) ──")
    p = sh(f'systemd-analyze verify "{UNIT}"', mostrar=False)
    saida = [l for l in (p.stdout + p.stderr).splitlines()
             if "atlas.service" in l]
    for linha in saida:
        print(f"   {linha}")
    if not saida:
        print("   ✅ nenhum problema de sintaxe")
    print("\n   💡 A queixa sobre o gunicorn é esperada: o caminho")
    print("      /opt/atlas/... não existe nesta máquina. Num servidor")
    print("      de verdade, ela apontaria um erro real.")
else:
    print("⚠️ systemd-analyze indisponível")

In [ ]:
referencia("""sudo cp atlas.service /etc/systemd/system/
sudo systemctl daemon-reload
sudo systemctl enable --now atlas
sudo systemctl status atlas""", """
● atlas.service - Atlas API — Aurora Comércio
     Loaded: loaded (/etc/systemd/system/atlas.service; enabled)
     Active: active (running) since Thu 2026-08-13 09:14:22 -03; 2min ago
   Main PID: 18432 (gunicorn)
      Tasks: 5 (limit: 4915)
     Memory: 187.4M
        CPU: 3.812s
""")

print()
comandos = [
    ["systemctl status atlas",        "estado atual"],
    ["systemctl restart atlas",       "🔴 derruba e sobe (há queda)"],
    ["systemctl reload atlas",        "✅ SIGHUP: recarrega sem queda"],
    ["journalctl -u atlas -f",        "🔧 log em tempo real"],
    ["journalctl -u atlas --since '10 min ago'", "log recente"],
    ["journalctl -u atlas -p err",    "só os erros"],
]
for cmd, desc in comandos:
    print(f"   {cmd:<44}{desc}")

print("\n🔑 `journalctl` é onde o seu log vai parar — porque o gunicorn")
print("   escreve em stdout (fator 11) e o systemd recolhe. Foi para")
print("   isso que você tirou o log do arquivo.")

## 7. 🔴 Migrações no deploy

In [ ]:
print("""
🔴 A PERGUNTA QUE DERRUBA SITE: MIGRAÇÃO ANTES OU DEPOIS DO CÓDIGO?

   Durante o deploy existe um momento em que código velho e código
   novo convivem — ou porque há várias instâncias, ou porque a
   antiga ainda está encerrando.

   ┌──────────────────────────────────────────────────────────┐
   │ MIGRAÇÃO ANTES                                            │
   │   O código VELHO precisa funcionar com o schema NOVO      │
   ├──────────────────────────────────────────────────────────┤
   │ MIGRAÇÃO DEPOIS                                           │
   │   O código NOVO precisa funcionar com o schema VELHO      │
   └──────────────────────────────────────────────────────────┘

🎯 A resposta prática: MIGRAÇÃO ANTES, e migrações COMPATÍVEIS.

   "Compatível" significa: o código velho continua funcionando com o
   schema novo. Isso é fácil para adicionar, difícil para remover.
""")

migracoes = [
    ["Adicionar coluna com default",  "✅ compatível",  "o código velho a ignora"],
    ["Adicionar tabela",              "✅ compatível",  "idem"],
    ["Adicionar índice",              "✅ compatível",  "⚠️ use CONCURRENTLY"],
    ["Renomear coluna",               "🔴 QUEBRA",      "o código velho procura o nome antigo"],
    ["Remover coluna",                "🔴 QUEBRA",      "o código velho ainda a lê"],
    ["Mudar tipo",                    "🔴 QUEBRA",      "e pode travar a tabela"],
    ["Adicionar NOT NULL sem default", "🔴 QUEBRA",     "o código velho insere NULL"],
]
tabela(["MIGRAÇÃO", "SEGURA?", "POR QUÊ"], migracoes, [32, 16, 40])

print("""
🔑 O PADRÃO DE TRÊS PASSOS (você viu no M05)

   Para renomear `nome` → `nome_completo`, sem queda:

   Deploy 1  · adiciona `nome_completo`
             · o código escreve nos DOIS, lê de `nome`
   Deploy 2  · o código lê de `nome_completo`
             · uma tarefa copia os dados que faltam
   Deploy 3  · o código para de escrever em `nome`
             · a migração remove `nome`

   Três deploys para renomear uma coluna. É chato — e é o preço de
   não derrubar o site. Em sistema sem usuário ativo, faça em um só.
""")

In [ ]:
print("""
⚠️ E ONDE A MIGRAÇÃO RODA?

   🔴 NÃO no start-up da aplicação. Com 4 instâncias subindo juntas,
      as 4 tentam migrar. O Alembic tem trava, mas o desenho está
      errado — e em Kubernetes isso vira reinício em laço.

   ✅ Num passo SEPARADO e ANTERIOR:
        · um serviço `migracao` no compose (M08), com
          `service_completed_successfully`
        · um `initContainer` no Kubernetes
        · um passo do pipeline, antes de trocar o symlink

   🔑 E ela precisa ser IDEMPOTENTE: rodar duas vezes não pode
      quebrar. O Alembic garante isso pela tabela de versão.
""")

referencia("""ssh atlas-prod '
    cd /opt/atlas/releases/20260813-9f8e7d
    .venv/bin/alembic upgrade head
'""", """
INFO  [alembic.runtime.migration] Running upgrade a1b2c3 -> d4e5f6, adiciona nome_completo
""")

## 8. 🎯 Rollback

In [ ]:
print("Estado antes do deploy problemático:\n")
situacao()

print(f"\n── deploy da versão {v3} ──")
publicar(v3, "VERSAO = 'nova'   # 🔴 e tem um bug")
situacao()

print("\n🔴 O erro apareceu. Rollback:\n")


def rollback():
    """Volta para o release anterior — em um comando.

    🔑 O `sorted` funciona porque o nome do release começa com a DATA
       em formato ISO. É o motivo de nomear assim, e não com um número
       sequencial ou um hash solto.
    """
    releases = sorted(p.name for p in (SERVIDOR / "releases").iterdir()
                      if p.is_dir())
    atual = os.readlink(SERVIDOR / "atual").split("/")[-1]
    if atual not in releases or releases.index(atual) == 0:
        raise RuntimeError("não há release anterior")
    anterior = releases[releases.index(atual) - 1]

    temporario = SERVIDOR / "atual.novo"
    if temporario.exists() or temporario.is_symlink():
        temporario.unlink()
    os.symlink(f"releases/{anterior}", temporario)
    os.replace(temporario, SERVIDOR / "atual")
    return anterior


voltou_para = rollback()
print(f"   revertido para {voltou_para}\n")
situacao()

print("""
🎯 O ROLLBACK LEVOU MILISSEGUNDOS.

   Não houve `git revert`, nem rebuild, nem redeploy. O release
   anterior já estava no disco, pronto — bastou apontar o symlink.

   💭 É por isso que se guarda os últimos N releases: o rollback é a
      operação que você faz sob pressão, com o site fora e o telefone
      tocando. Ela tem que ser trivial.
""")

In [ ]:
# Limpar releases antigos — mas não todos
def limpar(manter: int = 5):
    """Remove releases antigos, preservando os N mais recentes."""
    releases = sorted(p for p in (SERVIDOR / "releases").iterdir() if p.is_dir())
    atual = os.readlink(SERVIDOR / "atual").split("/")[-1]
    removidos = []
    for r in releases[:-manter]:
        # 🔴 Nunca remova o release que está ATIVO — mesmo que ele seja
        #    antigo (o que acontece exatamente depois de um rollback).
        if r.name == atual:
            continue
        shutil.rmtree(r)
        removidos.append(r.name)
    return removidos


print("releases antes:", sorted(p.name for p in (SERVIDOR / "releases").iterdir()))
print("removidos     :", limpar(manter=2))
print("releases depois:", sorted(p.name for p in (SERVIDOR / "releases").iterdir()))
print(f"ativo         : {os.readlink(SERVIDOR / 'atual').split('/')[-1]}")

print("\n⚠️ A checagem do release ATIVO não é teoria: logo após um")
print("   rollback, o release ativo é justamente um dos antigos.")
print("   Uma limpeza ingênua apagaria o que está no ar.")

> 🔴 **O rollback de código é fácil. O de banco, não.**
>
> Se o deploy problemático rodou uma migração que **removeu** uma coluna, voltar o código não devolve os dados. O `downgrade` do Alembic recria a coluna — vazia.
>
> 🧭 **Por isso as migrações compatíveis importam tanto:** com elas, o rollback do código funciona sozinho, porque o schema novo continua servindo o código velho.
>
> 💭 **A regra que se aprende com susto:** nunca destrua dado no mesmo deploy que muda o código. Separe em dois, com dias de distância — e um backup testado no meio.

## 9. Deploy sem queda — e seus limites

In [ ]:
print("""
   ═══ COM UMA INSTÂNCIA ═══

   1. envia o novo release
   2. roda a migração (compatível)
   3. troca o symlink            ← atômico
   4. systemctl reload atlas     ← SIGHUP: workers novos, sem queda

   ⚠️ O `reload` do gunicorn é gracioso: ele sobe workers novos, deixa
      os velhos terminarem o que estão fazendo, e só então os encerra.

   ═══ COM VÁRIAS INSTÂNCIAS ═══

   1. tira a instância A do balanceador
   2. espera as requisições em voo terminarem
   3. atualiza A e confere a saúde
   4. devolve A ao balanceador
   5. repete para B, C...

   💡 É o "rolling update". O Kubernetes faz isso sozinho; com Nginx
      você o implementa mexendo no `upstream`.
""")

print("""🔴 O QUE NENHUM DESSES RESOLVE

   · migração incompatível        → derruba de qualquer jeito
   · sessão em memória (09_01)    → o usuário cai ao trocar de worker
   · WebSocket aberto (M07)       → a conexão morre no reload
   · cache local frio             → picos de latência após o deploy

   💭 Deploy sem queda não é uma técnica de deploy. É uma propriedade
      da APLICAÇÃO — e você a construiu nos módulos anteriores.
""")

## 10. PaaS — quando não vale ter servidor

In [ ]:
opcoes = [
    ["VPS + systemd",   "R$ 30-200/mês",  "total",    "você cuida de tudo"],
    ["VPS + Docker",    "R$ 30-200/mês",  "total",    "+ ambiente reproduzível"],
    ["PaaS",            "R$ 0-500/mês",   "média",    "git push e pronto"],
    ["Kubernetes",      "R$ 400+/mês",    "total",    "🔶 só com equipe"],
    ["Serverless",      "por uso",        "baixa",    "⚠️ partida a frio"],
]
tabela(["OPÇÃO", "CUSTO", "CONTROLE", "OBSERVAÇÃO"], opcoes, [18, 18, 10, 30])

print("""
💭 A DECISÃO HONESTA PARA A AURORA

   Um e-commerce com uma pessoa de engenharia NÃO deveria começar com
   Kubernetes. O custo não é o cluster — é o seu tempo aprendendo e
   operando o cluster em vez de resolver as dores do negócio.

   🧭 A progressão sensata:
        PaaS ou VPS + Docker Compose  →  até doer
        aí sim, orquestrador          →  quando houver equipe

   ⚠️ E "até doer" tem sinais concretos: mais de 3 serviços com escala
      independente, necessidade de auto-escala, ou mais de uma pessoa
      fazendo deploy por dia.
""")

referencia("""# Fly.io
fly launch --dockerfile
fly secrets set ATLAS_SECRET_KEY=...
fly deploy""", """
==> Building image with Depot
--> build:  47.2s
==> Pushing image to fly
--> Done
==> Creating release
--> v3 deployed successfully
""")

print("\n💡 Repare: o PaaS usa o SEU Dockerfile (M08). O trabalho de")
print("   containerizar não é jogado fora — ele é justamente o que")
print("   permite trocar de plataforma sem reescrever nada.")

## 🔧 Prática guiada — o script de deploy

In [ ]:
DEPLOY = BASE / "deploy.sh"
DEPLOY.write_text("""#!/usr/bin/env bash
# ═══════════════════════════════════════════════════════════════
#  Deploy do Atlas
#      ./deploy.sh producao
# ═══════════════════════════════════════════════════════════════
set -euo pipefail
#   -e  aborta no primeiro erro   ← 🔴 essencial: sem isto, um passo
#       falha e o script continua, deixando o deploy pela metade
#   -u  variável não definida é erro
#   -o pipefail  erro no meio de um pipe não é engolido

AMBIENTE="${1:?uso: ./deploy.sh <producao|homologacao>}"
SERVIDOR="atlas-${AMBIENTE}"
RAIZ="/opt/atlas"
VERSAO="$(date -u +%Y%m%d-%H%M%S)-$(git rev-parse --short HEAD)"
RELEASE="${RAIZ}/releases/${VERSAO}"

echo "▶ deploy ${VERSAO} → ${AMBIENTE}"

# ── 1. Conferir ANTES de tocar no servidor ──
# 🔑 Falhar aqui custa 5 segundos. Falhar no servidor custa um
#    rollback e o site fora do ar.
git diff --quiet || { echo "🔴 há alterações não commitadas"; exit 1; }
pytest -q                                    # M07
python scripts/auditar_containers.py         # M08

# ── 2. Enviar ──
ssh "$SERVIDOR" "mkdir -p ${RELEASE}"
rsync -az --delete --exclude-from=deploy_excluir.txt \\
      ./ "${SERVIDOR}:${RELEASE}/"

# ── 3. Preparar o release ──
ssh "$SERVIDOR" bash -s <<REMOTO
  set -euo pipefail
  cd "${RELEASE}"
  ln -sfn ${RAIZ}/compartilhado/.env .env
  python -m venv .venv
  .venv/bin/pip install --no-cache-dir -q .
REMOTO

# ── 4. 🔴 Migrações ANTES de trocar o código ──
ssh "$SERVIDOR" "cd ${RELEASE} && .venv/bin/alembic upgrade head"

# ── 5. Trocar o symlink (atômico) e recarregar ──
ssh "$SERVIDOR" bash -s <<REMOTO
  set -euo pipefail
  ln -sfn "releases/${VERSAO}" "${RAIZ}/atual.novo"
  mv -T "${RAIZ}/atual.novo" "${RAIZ}/atual"
  sudo systemctl reload atlas
REMOTO

# ── 6. Conferir que subiu ──
# 🔴 Um deploy que não verifica não é um deploy — é uma esperança.
for i in $(seq 1 20); do
  if curl -fsS --max-time 3 https://atlas.aurora.com.br/saude >/dev/null; then
    echo "✅ ${VERSAO} no ar"
    ssh "$SERVIDOR" "ls -1dt ${RAIZ}/releases/*/ | tail -n +6 | xargs -r rm -rf"
    exit 0
  fi
  sleep 3
done

# ── 7. Não subiu: rollback automático ──
echo "🔴 saúde não respondeu — revertendo"
ssh "$SERVIDOR" "${RAIZ}/rollback.sh"
exit 1
""", encoding="utf-8")

p = sh(f'bash -n "{DEPLOY}"', mostrar=False)
print(f"✅ deploy.sh — sintaxe {'ok' if p.returncode == 0 else 'COM ERRO'}")
print(f"   {len(DEPLOY.read_text(encoding='utf-8').splitlines())} linhas\n")

for i, linha in enumerate(DEPLOY.read_text(encoding="utf-8").splitlines(), 1):
    if linha.startswith("# ── "):
        print(f"   L{i:<4}{linha[2:]}")

> 🎯 **Os dois passos que separam um script de deploy de um `git pull`:**
>
> **Passo 1 — conferir antes de sair de casa.** Testes e auditoria rodam na sua máquina. Falhar aqui custa 5 segundos; falhar no servidor custa um rollback.
>
> **Passo 6/7 — verificar e reverter sozinho.** Um deploy que não confere se a aplicação subiu não é um deploy: é uma esperança. E o rollback automático é o que transforma "o site está fora e não sei por quê" em "voltou sozinho em 60 segundos, agora eu investigo com calma".

In [ ]:
ROLLBACK = BASE / "rollback.sh"
ROLLBACK.write_text("""#!/usr/bin/env bash
# ═══════════════════════════════════════════════════════════════
#  Rollback do Atlas — roda NO SERVIDOR
#      /opt/atlas/rollback.sh
#
#  💭 Este script existe para ser rodado às 2h da manhã, por alguém
#     com sono, sob pressão. Por isso ele não tem opção nenhuma:
#     volta para o release anterior, e pronto.
# ═══════════════════════════════════════════════════════════════
set -euo pipefail

RAIZ="/opt/atlas"
ATUAL="$(basename "$(readlink "${RAIZ}/atual")")"
ANTERIOR="$(ls -1d "${RAIZ}"/releases/*/ | sed 's:/$::' | xargs -n1 basename \\
            | sort | grep -B1 "^${ATUAL}$" | head -1)"

if [ -z "$ANTERIOR" ] || [ "$ANTERIOR" = "$ATUAL" ]; then
    echo "🔴 não há release anterior"
    exit 1
fi

echo "revertendo ${ATUAL} → ${ANTERIOR}"
ln -sfn "releases/${ANTERIOR}" "${RAIZ}/atual.novo"
mv -T "${RAIZ}/atual.novo" "${RAIZ}/atual"
sudo systemctl reload atlas

sleep 3
curl -fsS --max-time 5 http://127.0.0.1:8000/saude >/dev/null \\
    && echo "✅ revertido para ${ANTERIOR}" \\
    || { echo "🔴 nem o release anterior responde — investigue o BANCO"; exit 1; }

# ⚠️ Se o release anterior TAMBÉM não responder, o problema quase
#    nunca é o código: é migração, banco fora, ou configuração.
""", encoding="utf-8")

p = sh(f'bash -n "{ROLLBACK}"', mostrar=False)
print(f"✅ rollback.sh — sintaxe {'ok' if p.returncode == 0 else 'COM ERRO'}")

print("\nEstrutura criada:\n")
arvore(BASE)

## 📝 Exercícios

**E1.** Gere um par ed25519 com senha e configure o `ssh-agent`. Explique o que a senha protege.

**E2.** Escreva um `~/.ssh/config` com dois hosts e explique `IdentitiesOnly`.

**E3.** 🔴 Liste as cinco linhas de endurecimento do `sshd_config` e explique cada uma. Diga por que manter uma segunda sessão aberta.

**E4.** Crie o usuário `deploy` e um `sudoers.d` que permita só reiniciar o serviço. Justifique.

**E5.** 🎯 Implemente `publicar()` e `rollback()` com symlink atômico. Prove que `os.replace` é atômico e `ln -sfn` não é.

**E6.** Explique por que o `.env` mora em `compartilhado/` e não no release.

**E7.** Monte uma lista de exclusões do rsync e meça a economia. Compare com o `.dockerignore` do M08.

**E8.** 🔴 Rode um `rsync --delete` com `--dry-run` apontando para o caminho errado. Descreva o que teria acontecido.

**E9.** Escreva um `atlas.service` e valide com `systemd-analyze verify`. Explique `TimeoutStopSec` vs `--graceful-timeout`.

**E10.** Explique a diferença entre `restart` e `reload` no systemd, e qual usar no deploy.

**E11.** 🔴 Classifique cinco migrações suas em compatíveis e incompatíveis. Para uma incompatível, escreva o plano de três deploys.

**E12.** Explique por que migração não deve rodar no start-up com várias instâncias.

**E13.** Implemente a limpeza de releases antigos preservando o ativo. Teste logo após um rollback.

**E14.** 🔴 Escreva o `deploy.sh` com verificação de saúde e rollback automático. Provoque a falha e veja reverter.

**E15.** Compare VPS, PaaS e Kubernetes para a Aurora. Escolha um e justifique com números.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```bash
# ═══ SSH ═══
ssh-keygen -t ed25519 -C "deploy@atlas"     # 🔑 ed25519, não RSA
ssh-copy-id -i ~/.ssh/atlas.pub deploy@servidor
# 🔴 /etc/ssh/sshd_config:
#    PermitRootLogin no · PasswordAuthentication no · AllowUsers deploy
# ⚠️ deixe uma 2ª sessão aberta antes de reiniciar o sshd

# ═══ 🎯 Deploy: releases + symlink ═══
/opt/atlas/
├── releases/20260813-9f8e7d/
├── compartilhado/.env          🔑 config e dados FORA do release
└── atual -> releases/...       🎯 um symlink

# troca ATÔMICA (ln -sfn NÃO é atômico)
ln -sfn releases/$V /opt/atlas/atual.novo
mv -T   /opt/atlas/atual.novo /opt/atlas/atual

# ═══ rsync ═══
rsync -az --delete --exclude-from=deploy_excluir.txt ./ servidor:/destino/
#     --dry-run   🔴 SEMPRE na primeira vez com --delete

# ═══ systemd ═══
systemctl reload atlas      ✅ SIGHUP, sem queda
systemctl restart atlas     🔴 derruba e sobe
journalctl -u atlas -f      🔧 o log (que veio do stdout)
systemd-analyze verify atlas.service
# TimeoutStopSec > --graceful-timeout do gunicorn

# ═══ 🔴 Migrações ═══
# ORDEM: migração ANTES do código, e COMPATÍVEL
# ✅ adicionar coluna/tabela/índice
# 🔴 renomear · remover · mudar tipo · NOT NULL sem default
# renomear sem queda = 3 deploys (escreve nos 2 → lê do novo → remove)
# 🔴 NUNCA no start-up da aplicação

# ═══ Ordem do deploy ═══
# 1. testar LOCAL   2. enviar   3. instalar   4. migrar
# 5. trocar symlink 6. reload   7. verificar  8. rollback se falhar
```

## ✅ Checklist de saída

**Acesso**

- [ ] 🔴 **Login por chave, nunca senha**
- [ ] `PermitRootLogin no`
- [ ] Usuário `deploy` sem privilégio, com sudo mínimo
- [ ] Sei manter uma segunda sessão ao mexer no sshd

**Deploy**

- [ ] 🎯 **Uso `releases/` + symlink**
- [ ] A troca do symlink é atômica (`mv -T`)
- [ ] Configuração e dados moram em `compartilhado/`
- [ ] Envio o artefato, não faço `git pull` no servidor
- [ ] `rsync` com exclusões
- [ ] Testo e audito **antes** de enviar

**Processo**

- [ ] `systemd` reinicia se cair
- [ ] `reload` em vez de `restart`
- [ ] `TimeoutStopSec` maior que o `--graceful-timeout`
- [ ] `StartLimitBurst` evita laço de reinício
- [ ] Log vai para o `journalctl`

**Banco**

- [ ] 🔴 **Migração antes do código, e compatível**
- [ ] Sei o plano de três deploys para renomear coluna
- [ ] Migração não roda no start-up
- [ ] Backup testado antes de migração destrutiva

**Rollback**

- [ ] 🔴 **Volta em um comando**
- [ ] O deploy reverte sozinho se a saúde falhar
- [ ] A limpeza de releases preserva o ativo
- [ ] Sei que rollback de banco é outro problema

---

### ➡️ Próxima aula

**`09_03_Pipelines_e_Monitoramento.ipynb`** — Tirar você do meio: o `git push` que testa, audita e publica sozinho. E o monitoramento que avisa antes do cliente.